In [8]:
import os
from uuid import uuid4
from dotenv import load_dotenv
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec

# Load environment variables from .env file
load_dotenv()

True

In [10]:
markdown_texts = """# Healthy Living Articles

## Benefits of Drinking Water
Drinking enough water every day helps maintain body temperature, protect tissues, and improve brain function. It also aids digestion and helps flush out toxins.

## Importance of Sleep
Sleep is crucial for physical and mental health. Lack of sleep can cause fatigue, poor concentration, and increased stress levels. Adults should aim for 7-9 hours.

## Eating a Balanced Diet
A healthy diet includes fruits, vegetables, whole grains, and healthy fats. Avoid processed foods high in sugar and salt.

# Travel Tips and Adventures

## Planning Your Trip
Start by researching your destination, local customs, and create a checklist for important documents such as passports, visas, and travel insurance.

## Packing Essentials
Pack light and include items like comfortable shoes, reusable water bottle, travel-sized toiletries, and a power bank. Always keep your essentials in your carry-on.

## Staying Safe
Keep digital copies of your important documents, don't share travel plans publicly and be cautious in crowded areas. Use hotel safes for valuables.
"""

In [11]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
chunks = splitter.split_text(markdown_texts)

print(f"Successfully split into {len(chunks)} chunks.")

Successfully split into 6 chunks.


In [12]:
# Initialize Google Gemini Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings


embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

index_name = 'testing-pinecone-gemini'

In [13]:
if index_name not in pc.list_indexes().names():
    print(f"Creating index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=768,  # Gemini embeddings are 768 dimensions
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
else:
    print(f"Index {index_name} already exists.")

index = pc.Index(index_name)

Index testing-pinecone-gemini already exists.


In [14]:
pinecone_vdb = PineconeVectorStore(index=index, embedding=embeddings)

uuids = [str(uuid4()) for _ in range(len(chunks))]

print("Adding documents to Pinecone...")
pinecone_vdb.add_documents(documents=chunks, ids=uuids)

Adding documents to Pinecone...


['750ede9e-6622-4f59-b0bb-6debadb38a53',
 'b0e63235-c570-4e66-803c-8e5cb5d74e6e',
 'b1f8ed5c-12eb-42d6-b905-0c012b2084c8',
 '72488590-5699-470d-bf2c-139f9cf2bc97',
 'eb310b54-ec44-4b2e-898f-1da3a0f257c9',
 '55c4d2f2-75c8-4ee8-8160-dc8ee5da8649']

In [15]:
print("\n--- Query 1: Travel Tips ---")
results = pinecone_vdb.similarity_search("What is the travel trips should i follow?", k=2)

for res in results:
    print(f"Content: {res.page_content}")

print("\n--- Query 2: Healthy Diet (With Scores) ---")
results_score = pinecone_vdb.similarity_search_with_score("What is healthy diet?", k=2)

for res, score in results_score:
    print(f"* [SIM={score:.3f}] {res.page_content} [{res.metadata}]")


--- Query 1: Travel Tips ---
Content: Start by researching your destination, local customs, and create a checklist for important documents such as passports, visas, and travel insurance.
Content: Start by researching your destination, local customs, and create a checklist for important documents such as passports, visas, and travel insurance.

--- Query 2: Healthy Diet (With Scores) ---
* [SIM=0.710] A healthy diet includes fruits, vegetables, whole grains, and healthy fats. Avoid processed foods high in sugar and salt. [{'Header 1': 'Healthy Living Articles', 'Header 2': 'Eating a Balanced Diet'}]
* [SIM=0.710] A healthy diet includes fruits, vegetables, whole grains, and healthy fats. Avoid processed foods high in sugar and salt. [{'Header 1': 'Healthy Living Articles', 'Header 2': 'Eating a Balanced Diet'}]
